In [1]:
import json
import time
import warnings
from pathlib import Path
import os
import mlflow
import lightgbm as lgb
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.metrics import average_precision_score, roc_auc_score

In [2]:
CURRENT_DIR = Path().cwd()
REPO_ROOT = CURRENT_DIR.parent
MODEL_VERSION = 3
DATA_DIR = REPO_ROOT / "dataset"
MODEL_DIR = REPO_ROOT / "models" / str(MODEL_VERSION)
MODEL_DIR.mkdir(exist_ok=True)

def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env_file(REPO_ROOT / ".env")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "")
MLFLOW_REGISTERED_MODEL_NAME = os.getenv(
    "MLFLOW_REGISTERED_MODEL_NAME", ""
)

GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID", "")
if GCP_PROJECT_ID:
    os.environ.setdefault("GOOGLE_CLOUD_PROJECT", GCP_PROJECT_ID)
    os.environ.setdefault("GCLOUD_PROJECT", GCP_PROJECT_ID)
    os.environ.setdefault("CLOUDSDK_CORE_PROJECT", GCP_PROJECT_ID)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

<Experiment: artifact_location='gs://fraud-detection-aide/mlflow-artifacts/1', creation_time=1779635430697, experiment_id='1', last_update_time=1779635430697, lifecycle_stage='active', name='fraud-detection-aide', tags={}, trace_location=None, workspace='default'>

In [3]:
if not os.path.exists(DATA_DIR):
    print("Dataset not found. Please download dataset on Kaggle.")
else:
    print("Dataset found. Continuing...")

Dataset found. Continuing...


In [4]:
SEED = 42
TARGET = "label"

LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 256,
    "min_child_samples": 80,
    "feature_fraction": 0.5,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "n_jobs": -1,
    "seed": SEED,
}
NUM_BOOST_ROUND = 5000
EARLY_STOPPING = 200
TOP_K_FEATURES = 250       
VAL_FRAC = 0.15            
FINAL_ROUND_BUMP = 1.10    

UID_COLS = ["uid1", "uid2", "uid3", "uid4"]
UID_AGG_TARGETS = ["amount_usd", "C13", "D15", "D4"]


RENAME = {
    "TransactionID":  "tx_id",
    "TransactionDT":  "event_ts_offset_s",
    "TransactionAmt": "amount_usd",
    "ProductCD":      "channel",
    "isFraud":        "label",
    "card1":          "card_id",
    "card2":          "issuer_code",
    "card3":          "card_country",
    "card4":          "card_brand",
    "card5":          "bin_code",
    "card6":          "card_type",
    "addr1":          "billing_zone",
    "addr2":          "billing_country",
    "P_emaildomain":  "email_purchaser",
    "R_emaildomain":  "email_recipient",
    "D1":             "card_age_days",
    "D15":            "days_since_last_tx",
    "DeviceType":     "device_type",
    "DeviceInfo":     "device_info",
    "id_30":          "os_raw",
    "id_31":          "browser_raw",
    "id_33":          "screen_resolution",
}

EMAIL_BIN = {
    "gmail.com": "google", "googlemail.com": "google",
    "yahoo.com": "yahoo", "ymail.com": "yahoo", "rocketmail.com": "yahoo",
    "hotmail.com": "microsoft", "outlook.com": "microsoft",
    "live.com": "microsoft", "msn.com": "microsoft",
    "aol.com": "aol", "aim.com": "aol",
    "icloud.com": "apple", "me.com": "apple", "mac.com": "apple",
}
EMAIL_NULLS = {"anonymous.com", "mail.com"}

In [5]:
def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

In [6]:
def load_split(name: str) -> pd.DataFrame:
    log(f"Loading {name} transaction + identity ...")
    tx = pd.read_csv(DATA_DIR / f"{name}_transaction.csv")
    idn = pd.read_csv(DATA_DIR / f"{name}_identity.csv")
    idn.columns = [c.replace("id-", "id_") for c in idn.columns]
    df = tx.merge(idn, on="TransactionID", how="left")
    df = df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns})
    log(f"  {name} shape: {df.shape}, identity coverage: "
        f"{idn['TransactionID'].nunique() / len(tx):.1%}")
    return df

def _device_brand(s: str) -> str:
    s = s.lower().strip()
    if not s or s == "nan":
        return "missing"
    head = s.split("/")[0].split(" ")[0]
    if head.startswith("sm-") or "samsung" in s:
        return "samsung"
    for needle in ("moto", "lg", "huawei", "xiaomi", "oppo", "vivo",
                   "redmi", "nokia", "lenovo", "asus", "htc", "google",
                   "trident", "rv:", "windows", "macos", "linux", "ios",
                   "ipad", "iphone"):
        if needle in s:
            return needle
    return head[:12] or "other"


def _os_family(s: str) -> str:
    s = s.lower()
    for k in ("windows", "ios", "android", "mac", "linux"):
        if k in s:
            return k
    return "other" if s and s != "nan" else "missing"


def _browser_family(s: str) -> str:
    s = s.lower()
    for k in ("chrome", "safari", "firefox", "edge", "opera",
              "samsung", "ie ", "android"):
        if k in s:
            return k.strip()
    return "other" if s and s != "nan" else "missing"


def add_identity_features(df: pd.DataFrame) -> None:
    if "device_info" in df.columns:
        di = df["device_info"].fillna("missing").astype(str)
        df["device_info"] = di
        df["device_brand"] = di.map(_device_brand)

    if "os_raw" in df.columns:
        s = df["os_raw"].fillna("missing").astype(str)
        df["os_raw"] = s
        df["os_family"] = s.map(_os_family)
        df["os_version"] = pd.to_numeric(
            s.str.extract(r"(\d+(?:\.\d+)?)", expand=False), errors="coerce"
        ).astype(np.float32)

    if "browser_raw" in df.columns:
        s = df["browser_raw"].fillna("missing").astype(str)
        df["browser_raw"] = s
        df["browser_family"] = s.map(_browser_family)
        df["browser_version"] = pd.to_numeric(
            s.str.extract(r"(\d+(?:\.\d+)?)", expand=False), errors="coerce"
        ).astype(np.float32)

    if "screen_resolution" in df.columns:
        s = df["screen_resolution"].fillna("0x0").astype(str)
        wh = s.str.split("x", n=1, expand=True)
        df["screen_width"]  = pd.to_numeric(wh[0], errors="coerce").astype(np.float32)
        df["screen_height"] = pd.to_numeric(wh[1], errors="coerce").astype(np.float32)
        df["screen_area"]   = (df["screen_width"] * df["screen_height"]).astype(np.float32)

    if "device_type" in df.columns:
        df["device_type"] = df["device_type"].fillna("missing").astype(str)


In [7]:
FREQ_COLS = [
    "card_id", "issuer_code", "bin_code",
    "billing_zone", "billing_country",
    "email_purchaser", "email_recipient",
    "uid1", "uid2", "uid3", "uid4",
    "device_info", "device_brand", "os_raw", "browser_raw",
]

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    log("Feature engineering ...")
    ref = pd.Timestamp("2017-12-01")
    ts = ref + pd.to_timedelta(df["event_ts_offset_s"], unit="s")
    df["hour_of_day"] = ts.dt.hour.astype(np.int8)
    df["day_of_week"] = ts.dt.dayofweek.astype(np.int8)
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(np.int8)

    df["amount_log"]   = np.log1p(df["amount_usd"]).astype(np.float32)
    df["amount_cents"] = (df["amount_usd"] - df["amount_usd"].astype(int)).astype(np.float32)

    add_identity_features(df)

    for col in ["email_purchaser", "email_recipient"]:
        df[col] = df[col].where(~df[col].isin(EMAIL_NULLS), other=np.nan)
        df[f"{col}_provider"] = df[col].map(EMAIL_BIN).fillna("other")
        df[col] = df[col].fillna("missing").astype(str)

    day = (df["event_ts_offset_s"] // 86400).astype(int)
    first_seen_day = (day - df["card_age_days"].fillna(0).astype(int)).astype(str)

    def _s(col: str) -> pd.Series:
        return df[col].fillna("missing").astype(str)

    df["uid1"] = _s("card_id") + "_" + _s("billing_zone")
    df["uid2"] = df["uid1"] + "_" + first_seen_day
    df["uid3"] = df["uid2"] + "_" + _s("email_purchaser")
    brand = _s("device_brand") if "device_brand" in df.columns else "missing"
    df["uid4"] = df["uid3"] + "_" + brand

    for uid in UID_COLS:
        for col in UID_AGG_TARGETS:
            if col not in df.columns:
                continue
            grp = df.groupby(uid, dropna=False)[col]
            df[f"{uid}_{col}_mean"] = grp.transform("mean").astype(np.float32)
            df[f"{uid}_{col}_std"]  = grp.transform("std").astype(np.float32)

    df = df.sort_values("event_ts_offset_s").reset_index(drop=True)
    g = df.groupby("uid2", dropna=False)
    df["card_tx_count_so_far"] = g.cumcount().astype(np.int32)
    cum_amt = g["amount_usd"].cumsum().astype(np.float32) - df["amount_usd"]
    df["card_amount_sum_so_far"]  = cum_amt
    df["card_amount_mean_so_far"] = (
        cum_amt / df["card_tx_count_so_far"].replace(0, np.nan)
    ).astype(np.float32)
    df["amount_zscore_card"] = (
        (df["amount_usd"] - df["card_amount_mean_so_far"])
        / df["card_amount_mean_so_far"].replace(0, np.nan)
    ).astype(np.float32)

    log(f"  features after FE: {df.shape[1]}")
    return df


In [8]:
def freq_encode(df: pd.DataFrame) -> dict[str, dict[str, int]]:
    tables: dict[str, dict[str, int]] = {}
    for col in FREQ_COLS:
        if col not in df.columns:
            continue
        counts = df[col].astype(str).value_counts(dropna=False)
        tables[col] = counts.to_dict()
        df[f"{col}_freq"] = df[col].astype(str).map(counts).astype(np.float32)
    df.drop(columns=UID_COLS, inplace=True, errors="ignore")
    return tables


def encode_categoricals(df: pd.DataFrame) -> tuple[list[str], dict[str, dict[str, int]]]:
    skip = {TARGET, "tx_id", "event_ts_offset_s"}
    obj_cols = [
        c for c in df.columns
        if c not in skip
        and not pd.api.types.is_numeric_dtype(df[c])
        and not pd.api.types.is_bool_dtype(df[c])
    ]
    encoders: dict[str, dict[str, int]] = {}
    for col in obj_cols:
        s = df[col].astype(str).fillna("missing")
        mapping = {v: i for i, v in enumerate(s.unique())}
        df[col] = s.map(mapping).astype(np.int32)
        encoders[col] = mapping
    log(f"Label-encoded {len(obj_cols)} categorical columns.")
    return obj_cols, encoders


def temporal_split(df: pd.DataFrame, val_frac: float = VAL_FRAC) -> tuple[np.ndarray, np.ndarray]:
    order = df["event_ts_offset_s"].argsort().values
    n = len(order)
    cut = int(n * (1.0 - val_frac))
    tr, va = order[:cut], order[cut:]
    log(f"Split — train: {len(tr):,}  val: {len(va):,} (last {val_frac:.0%})")
    return tr, va


def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k_frac: float) -> float:
    k = max(1, int(len(y_score) * k_frac))
    top = np.argpartition(-y_score, k - 1)[:k]
    return float(y_true[top].sum()) / k

In [9]:
def select_features(
    train_df: pd.DataFrame, cat_cols: list[str], top_k: int
) -> list[str]:
    drop_cols = {TARGET, "tx_id", "event_ts_offset_s"}
    all_feats = [c for c in train_df.columns if c not in drop_cols]
    if len(all_feats) <= top_k:
        log(f"Have {len(all_feats)} features (<= {top_k}); skipping selection.")
        return all_feats

    log(f"Feature selection: ranking {len(all_feats)} features, keeping top {top_k} ...")
    X = train_df[all_feats]
    y = train_df[TARGET].astype(np.int8).values
    tr, va = temporal_split(train_df)
    cat_idx = [c for c in cat_cols if c in all_feats]

    pos = y[tr].sum()
    fast = {**LGB_PARAMS, "learning_rate": 0.05,
            "scale_pos_weight": (len(tr) - pos) / max(1, pos)}
    dtr = lgb.Dataset(X.iloc[tr], label=y[tr], categorical_feature=cat_idx)
    dva = lgb.Dataset(X.iloc[va], label=y[va], categorical_feature=cat_idx)
    rank = lgb.train(
        fast, dtr,
        num_boost_round=2000,
        valid_sets=[dva], valid_names=["valid"],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(period=300)],
    )
    gains = pd.Series(
        rank.feature_importance(importance_type="gain"), index=all_feats
    ).sort_values(ascending=False)
    selected = gains.head(top_k).index.tolist()
    log(f"  kept {len(selected)} features  (min gain={gains.iloc[top_k-1]:.1f}, "
        f"dropped {len(all_feats) - len(selected)})")
    return selected


In [10]:
def train_phase_a(
    train_df: pd.DataFrame, feat_cols: list[str], cat_cols: list[str]
) -> tuple[int, dict]:
    log(f"Phase A — fit on first {1-VAL_FRAC:.0%}, early-stop on last {VAL_FRAC:.0%}.")
    X = train_df[feat_cols]
    y = train_df[TARGET].astype(np.int8).values
    tr, va = temporal_split(train_df)
    cat_idx = [c for c in cat_cols if c in feat_cols]

    pos = y[tr].sum()
    pos_weight = (len(tr) - pos) / max(1, pos)
    params = {**LGB_PARAMS, "scale_pos_weight": pos_weight}

    dtr = lgb.Dataset(X.iloc[tr], label=y[tr], categorical_feature=cat_idx)
    dva = lgb.Dataset(X.iloc[va], label=y[va], categorical_feature=cat_idx)
    booster = lgb.train(
        params, dtr,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dva], valid_names=["valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING), lgb.log_evaluation(period=200)],
    )
    val_pred = booster.predict(X.iloc[va], num_iteration=booster.best_iteration)
    metrics = {
        "val_roc_auc":           float(roc_auc_score(y[va], val_pred)),
        "val_pr_auc":            float(average_precision_score(y[va], val_pred)),
        "val_precision_at_1pct": precision_at_k(y[va], val_pred, 0.01),
        "best_iteration":        int(booster.best_iteration),
        "n_features":            len(feat_cols),
        "n_train_phase_a":       int(len(tr)),
        "scale_pos_weight":      float(pos_weight),
    }
    log("Phase A metrics:\n" + json.dumps(metrics, indent=2))
    return booster.best_iteration, metrics


def train_phase_b(
    train_df: pd.DataFrame, feat_cols: list[str], cat_cols: list[str], best_iter: int
) -> tuple[lgb.Booster, int]:
    n_rounds = max(50, int(best_iter * FINAL_ROUND_BUMP))
    log(f"Phase B — refit on FULL labeled set ({len(train_df):,} rows) "
        f"for {n_rounds} rounds (best_iter {best_iter} x {FINAL_ROUND_BUMP}).")
    X = train_df[feat_cols]
    y = train_df[TARGET].astype(np.int8).values
    cat_idx = [c for c in cat_cols if c in feat_cols]

    pos = y.sum()
    params = {**LGB_PARAMS, "scale_pos_weight": (len(y) - pos) / max(1, pos)}
    dtr = lgb.Dataset(X, label=y, categorical_feature=cat_idx)
    booster = lgb.train(
        params, dtr,
        num_boost_round=n_rounds,
        callbacks=[lgb.log_evaluation(period=200)],
    )
    booster.save_model(MODEL_DIR / "model.bst")
    return booster, n_rounds


In [11]:
# train = load_split("train")
# test  = load_split("test")

# n_train = len(train)
# full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)
# del train, test

# full = build_features(full)
# freq_tables = freq_encode(full)
# cat_cols, encoders = encode_categoricals(full)

# train_df = full.iloc[:n_train].reset_index(drop=True)
# test_df  = full.iloc[n_train:].reset_index(drop=True)
# assert train_df[TARGET].notna().all(), "train rows must have label"
# del full

# feat_cols = select_features(train_df, cat_cols, TOP_K_FEATURES)
# cat_idx = [c for c in cat_cols if c in feat_cols]

# best_iter, val_metrics = train_phase_a(train_df, feat_cols, cat_cols)
# model, n_rounds = train_phase_b(train_df, feat_cols, cat_cols, best_iter)

# metrics = {**val_metrics, "phase_b_n_rounds": n_rounds,
#             "n_train_phase_b": int(len(train_df))}

# schema = {
#     "version": 1,
#     "training_reference_ts": "2017-12-01",
#     "rename_at_ingest": RENAME,
#     "email_bin": EMAIL_BIN,
#     "email_nulls": sorted(EMAIL_NULLS),
#     "uid_columns": UID_COLS,
#     "uid_agg_targets": UID_AGG_TARGETS,
#     "freq_columns": FREQ_COLS,
#     "freq_tables": freq_tables,
#     "categorical_features": cat_idx,
#     "categorical_encoders": encoders,
#     "feature_columns": feat_cols,
#     "target": TARGET,
# }

In [12]:
metrics_path = MODEL_DIR / "metrics.json"
schema_path = MODEL_DIR / "feature_schema.json"
feature_importance_path = MODEL_DIR / "feature_importance.csv"
model_path = MODEL_DIR / "model.bst"
import json 
with open(metrics_path, "r") as f:
    metrics = json.load(f)

In [13]:
import lightgbm
model = lightgbm.Booster(model_file=str(model_path))

In [14]:
with mlflow.start_run(run_name=f"lightgbm-v{MODEL_VERSION}") as run:
    mlflow.log_params({
        "model_version": MODEL_VERSION,
        "model_type": "lightgbm",
    })
    mlflow.log_params({f"lgb_{key}": value for key, value in LGB_PARAMS.items()})
    mlflow.log_metrics({key: float(value) for key, value in metrics.items()})
    mlflow.log_artifact(model_path)
    mlflow.log_artifact(metrics_path)
    mlflow.log_artifact(schema_path)
    mlflow.log_artifact(feature_importance_path)

    model_info = mlflow.lightgbm.log_model(
        model,
        name="model",
        registered_model_name=MLFLOW_REGISTERED_MODEL_NAME,
    )

client = MlflowClient()
registered_versions = client.search_model_versions(
    f"name = '{MLFLOW_REGISTERED_MODEL_NAME}' and run_id = '{run.info.run_id}'"
)
if not registered_versions:
    raise RuntimeError("MLflow did not create a registered model version for this run")

registered_version = registered_versions[0]
if int(registered_version.version) != MODEL_VERSION:
    raise RuntimeError(
        f"Expected registered model version {MODEL_VERSION}, "
        f"got {registered_version.version}. Delete the existing registered model "
        f"'{MLFLOW_REGISTERED_MODEL_NAME}' or use a new name to create version 1."
    )

client.set_model_version_tag(
    name=MLFLOW_REGISTERED_MODEL_NAME,
    version=str(MODEL_VERSION),
    key="model_version",
    value=str(MODEL_VERSION),
)

log(
    f"Registered {MLFLOW_REGISTERED_MODEL_NAME} version "
    f"{registered_version.version}: {model_info.model_uri}"
)

2026/05/26 22:31:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'fraud-detection-lightgbm' already exists. Creating a new version of this model...
2026/05/26 22:31:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-lightgbm, version 2
Created version '2' of model 'fraud-detection-lightgbm'.


🏃 View run lightgbm-v3 at: http://localhost:5000/#/experiments/1/runs/f2b0ac5e510b4a8889e9d5131b685ac4
🧪 View experiment at: http://localhost:5000/#/experiments/1


RuntimeError: Expected registered model version 3, got 2. Delete the existing registered model 'fraud-detection-lightgbm' or use a new name to create version 1.